Copyright 2018 The TensorFlow Datasets Authors, Licensed under the Apache License, Version 2.0

In [2]:
!pip install -q tensorflow-datasets tensorflow

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

# Construct a tf.data.Dataset
ds = tfds.load('mnist', split='train', shuffle_files=True)

# Build your input pipeline
ds = ds.shuffle(1024).batch(32).prefetch(tf.data.AUTOTUNE)
for example in ds.take(1):
  image, label = example['image'], example['label']

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.Y9JTIW_3.0.1/mnist-train.tfrecord*...:   0%|          | 0…

Generating test examples...: 0 examples [00:00, ? examples/s]

### 1. 載入必要的函式庫與設定路徑
我們將使用 TensorFlow 和 Keras 來建立模型。

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os

# 設定基礎路徑與參數
base_dir = 'ancient_animals_dataset'
img_height, img_width = 224, 224
batch_size = 32

### 2. 準備資料集
從目錄中載入圖片，並將其分為訓練集與驗證集。

In [ ]:
import os
import shutil

# 1. 確保資料夾結構與圖片存在
base_dir = '/content/ancient_animals_dataset'
classes = ['trilobite', 't_rex', 'mammoth', 'stegosaurus', 'pterodactyl']
for cls in classes:
    os.makedirs(os.path.join(base_dir, cls), exist_ok=True)

# 2. 列出目前已有的檔案
all_files = []
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_files.append(os.path.join(root, file))
print(f'目前資料集中的圖片總數: {len(all_files)}')
for f in all_files[:3]: print(f'範例路徑: {f}')

# 3. 載入資料集
try:
    if len(all_files) > 0:
        train_ds = tf.keras.utils.image_dataset_from_directory(
          base_dir,
          image_size=(img_height, img_width),
          batch_size=batch_size,
          shuffle=True)

        class_names = train_ds.class_names
        val_ds = train_ds
        print(f'成功載入！分類標籤: {class_names}')
    else:
        print('錯誤：目錄中完全沒有圖片，請確認下載步驟是否成功。')
except Exception as e:
    print(f'載入失敗：{e}')

In [ ]:
import os
import shutil

# 協助將上傳的圖片移動到正確的類別資料夾以供訓練
# 請根據您上傳的檔案名稱與類別修改下方的變數
image_moves = {
    'nt (1).JPG': 'stegosaurus',
    # '其他檔名.jpg': '類別名稱'
}

for src_file, class_name in image_moves.items():
    source = f'/content/{src_file}'
    destination = f'/content/ancient_animals_dataset/{class_name}/{src_file}'

    if os.path.exists(source):
        shutil.move(source, destination)
        print(f'已將 {src_file} 移動至 {class_name} 資料夾')
    else:
        print(f'找不到檔案: {src_file}')

In [ ]:
# 使用 Data Augmentation 增加訓練多樣性
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.1),
])

# 修改模型架構，加入 Data Augmentation 層
model = models.Sequential([
  layers.Input(shape=(img_height, img_width, 3)),
  data_augmentation,
  base_model,
  layers.GlobalAveragePooling2D(),
  layers.Dropout(0.2),
  layers.Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('模型已更新，加入資料增強層以提升泛化能力。')

In [ ]:
import tensorflow as tf

# 重新訓練模型 (建議在確認資料集已載入後執行)
try:
    print(f'開始訓練... 使用分類: {class_names}')
    history = model.fit(
        train_ds,
        epochs=10,
        verbose=1
    )
    print('\n訓練完成！您現在可以使用後續的預測函式來測試模型了。')
except Exception as e:
    print(f'訓練失敗：{e}')
    print('提示：請確保執行過「載入資料集」儲存格，且每個類別資料夾內至少有一張圖片。')

### 3. 建立模型 (遷移學習)
使用 MobileNetV2 作為基礎模型以提高效率。

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 更換為 EfficientNetV2L 模型
print('正在載入 EfficientNetV2L...')
base_model = tf.keras.applications.EfficientNetV2L(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)

# 開放基礎模型進行微調
base_model.trainable = True

# 重新建立模型結構
model = models.Sequential([
  base_model,
  layers.GlobalAveragePooling2D(),
  layers.Dropout(0.3), # 稍微增加 Dropout 以防止過擬合
  layers.Dense(len(class_names), activation='softmax')
])

# 使用較低的學習率進行微調
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print('\n模型已更換為 EfficientNetV2L，準備好進行訓練。')

### 4. 訓練與預測範例
訓練模型並撰寫一個函式來辨識新圖片。

In [ ]:
def predict_ancient_animal(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=(img_height, img_width))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    print(f'這張圖片最可能是 {class_names[np.argmax(score)]} (信心度: {100 * np.max(score):.2f}%)')
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
import tensorflow as tf
import os

# 使用 TensorFlow 內建工具下載圖片
test_url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'

try:
    print(f'正在下載測試圖片...')
    # 下載並解壓一個更可靠的來源，或者直接下載單張圖片
    path_to_img = tf.keras.utils.get_file('test_flower.jpg', 'https://storage.googleapis.com/download.tensorflow.org/example_images/fregate_bird.jpg')

    print(f'圖片下載成功，路徑為: {path_to_img}')

    # 執行預測
    predict_ancient_animal(path_to_img)
except Exception as e:
    print(f'發生錯誤: {e}')
    print('若網路持續阻擋下載，請點擊左側資料夾圖示手動上傳圖片，並執行 predict_ancient_animal("您的圖片路徑")')

In [ ]:
import tensorflow as tf
import os
import requests
import matplotlib.pyplot as plt

# 猛獁象測試圖片網址 (來自 Wikimedia)
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/0d/Mammoth_MG_3482.jpg/640px-Mammoth_MG_3482.jpg'
test_img_path = '/content/mammoth_test.jpg'

try:
    print('正在使用 requests 模擬瀏覽器下載圖片...')
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    with open(test_img_path, 'wb') as f:
        f.write(response.content)

    if os.path.exists(test_img_path):
        print(f'下載成功！圖片已儲存至: {test_img_path}')
        # 執行預測
        predict_ancient_animal(test_img_path)
    else:
        print('檔案儲存失敗。')

except Exception as e:
    print(f'下載失敗：{e}')
    print('請確認網路連線是否正常。')


In [ ]:
import numpy as np
from PIL import Image

# 由於網路下載受限，我們建立一張隨機圖片來測試預測函式
synthetic_img_path = '/content/synthetic_mammoth.jpg'

# 建立 224x224 的隨機 RGB 圖片
random_img_array = np.random.randint(0, 255, (img_height, img_width, 3), dtype=np.uint8)
img = Image.fromarray(random_img_array)
img.save(synthetic_img_path)

print(f'已生成虛擬測試圖片: {synthetic_img_path}')

try:
    # 測試預測函式
    predict_ancient_animal(synthetic_img_path)
    print('\n成功！預測流程測試完成。雖然圖片是隨機的，但這證明了模型架構與預測程式碼均可正常運作。')
except Exception as e:
    print(f'預測流程測試失敗: {e}')

### 5. 手動上傳並辨識圖片
執行下方儲存格來上傳您的圖片。

In [ ]:
from google.colab import files
import os

# 啟動上傳介面
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'正在辨識上傳的檔案: {filename}')
    # 執行預測
    predict_ancient_animal(filename)